# 21 — Temporal Split Design

**Objetivo:** congelar una validación verdaderamente temporal. No se usa `train_test_split` aleatorio.

In [6]:
from argos.common.paths import ProjectPaths
from argos.config.gold import load_gold_spec

paths = ProjectPaths.discover()
paths.ensure_runtime_dirs()

m2_gate = paths.docs / "decisions" / "M2_GATE.md"
assert m2_gate.exists() and "M2 ✅ CLOSED" in m2_gate.read_text(encoding="utf-8"), (
    "M2 no está formalmente cerrado. No iniciar Gold."
)

GOLD_SPEC = load_gold_spec(paths.configs / "gold.toml")

{
    "gold_schema_version": GOLD_SPEC.schema_version,
    "target": GOLD_SPEC.target.name,
    "horizon_days": GOLD_SPEC.target.horizon_days,
    "event_scope": GOLD_SPEC.target.included_event_types,
    "duckdb_threads": GOLD_SPEC.runtime.duckdb_threads,
    "duckdb_memory_limit": GOLD_SPEC.runtime.duckdb_memory_limit,
    "m2_gate": str(m2_gate),
}

from argos.gold.splits import BUILD_IMPLEMENTATION
BUILD_IMPLEMENTATION

'm3-21-temporal-split-v1'

## Política congelada

- Train: 2015–2023
- Validation: 2024
- Test: 2025
- 2026: elegible pero fuera del split principal, porque el año está parcialmente observado en el snapshot actual.

El test queda congelado para evaluación final y no se usa para feature engineering ni tuning.

In [7]:
from argos.gold.splits import build_temporal_splits

result = build_temporal_splits(
    gold_dir=paths.gold,
    reports_root=paths.reports,
    manifest_dir=paths.root / "artifacts" / "manifests",
    gold_spec=GOLD_SPEC,
    temp_directory=paths.root / ".cache" / "duckdb",
    overwrite=True,
)

str(result.artifact.relative_to(paths.root))

[21] materializing frozen chronological split assignments...


'data/gold/splits/temporal_split.parquet'

In [8]:
display(result.summary)
display(result.by_year)

,split,row_count,positives,negatives,min_t0,max_t0,prevalence
0,train,2781601,58446,2723155,2015-06-11,2023-12-31,0.021012
1,validation,950670,9180,941490,2024-01-01,2024-12-31,0.009656
2,test,1050857,10983,1039874,2025-01-01,2025-12-31,0.010451
3,eligible_out_of_split,529054,7255,521799,2026-01-01,2026-03-13,0.013713
4,not_eligible,754548,0,0,2026-03-14,2026-09-19,NaN


,t0_year,split,row_count,positives,negatives
0,2015,train,10,1,9
1,2016,train,1342,19,1323
2,2017,train,22259,859,21400
3,2018,train,142973,9474,133499
4,2019,train,142592,8036,134556
5,2020,train,357251,9682,347569
6,2021,train,561581,5803,555778
7,2022,train,710534,9289,701245
8,2023,train,843059,15283,827776
9,2024,validation,950670,9180,941490


In [9]:
result.validation

,check_id,category,blocking,passed,observed,requirement
0,split_periods_ordered,temporal,True,True,"{""test"": [""2025-01-01"", ""2025-12-31""], ""train""...",train < validation < test with no temporal ove...
1,train_population_sufficient,split,True,True,"{""negatives"": 2723155, ""positives"": 58446, ""ro...","train has >= 100000 rows, >= 500 positives and..."
2,validation_population_sufficient,split,True,True,"{""negatives"": 941490, ""positives"": 9180, ""rows...","validation has >= 50000 rows, >= 500 positives..."
3,test_population_sufficient,split,True,True,"{""negatives"": 1039874, ""positives"": 10983, ""ro...","test has >= 50000 rows, >= 500 positives and n..."
4,assigned_rows_labeled,target,True,True,"{""assigned_unlabeled"": 0}",all train/validation/test rows are eligible an...
5,split_grain_unique,grain,True,True,"{""duplicate_excess_rows"": 0}",one split assignment per contract
6,test_strictly_future,temporal,True,True,"{""test_min_t0"": ""2025-01-01 00:00:00"", ""valida...",test period starts strictly after validation ends


In [10]:
blocking = result.validation.loc[result.validation["blocking"]]
print("21 ✅ PASS" if result.ready_for_next_notebook else "21 ❌ FAIL")
print(f"Blocking checks: {int(blocking['passed'].sum())}/{len(blocking)}")
print("Manifest:", result.manifest_path)
assert result.ready_for_next_notebook, "Split temporal no supera sus checks bloqueantes."

21 ✅ PASS
Blocking checks: 7/7
Manifest: /Users/macm1pro/Desktop/projects/argos/argos-code/artifacts/manifests/gold_temporal_split.json
